# Gradient Boosting Regression Practice

**Topic:** Gradient Boosting Intuition &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `X_train` `(4000, 8)`, `y_train` `(4000,)` (house prices driven by nonlinear effects: diminishing returns on size, a sharp premium threshold on location score, an age×renovation interaction, plus noise).
- `X_test` `(1200, 8)`: predict prices, and `y_test` is used by the evaluation cell.

## Your task

1. Fit a gradient-boosted regressor (`HistGradientBoostingRegressor` or `GradientBoostingRegressor`).
2. Tune `learning_rate` and the number of iterations against a validation split. Print the validation score and the number of iterations for each setting you try, and note the classic tradeoff between the two.
3. Set `predictions` to your predicted prices for `X_test`.

## Evaluation

Test R² must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

def make(n, seed):
    r = np.random.default_rng(seed)
    size = r.gamma(6.0, 20.0, n)              # sq meters
    loc = r.uniform(0, 10, n)                 # location score
    age = r.uniform(0, 60, n)
    reno = (r.random(n) < 0.3).astype(float)  # renovated?
    rooms = r.poisson(3.0, n) + 1
    dist = r.gamma(2.0, 4.0, n)               # km to center
    crime = r.uniform(0, 5, n)
    garden = (r.random(n) < 0.5).astype(float)
    price = (
        40 * np.sqrt(size)
        + 55 * (loc > 7.0) + 6 * loc
        - 1.1 * age + 0.9 * age * reno + 18 * reno
        + 9 * rooms - 4.5 * dist - 6 * crime + 12 * garden
        + r.normal(0, 22, n)
    )
    X = np.column_stack([size, loc, age, reno, rooms, dist, crime, garden])
    return X, price

X_train, y_train = make(4000, 91)
X_test, y_test = make(1200, 92)
print(f"X_train {X_train.shape}, price range [{y_train.min():.0f}, {y_train.max():.0f}]")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_R2 = 0.85

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions, dtype=float).ravel()
_true = np.asarray(y_test, dtype=float).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

_rmse = float(np.sqrt(np.mean((_pred - _true) ** 2)))
_ss_res = float(np.sum((_true - _pred) ** 2))
_ss_tot = float(np.sum((_true - _true.mean()) ** 2))
r2 = 1.0 - _ss_res / _ss_tot
print("=" * 52)
print(f"Test RMSE : {_rmse:.4f}")
print(f"Test R^2  : {r2:.4f}   (target: >= {TARGET_R2:.2f})")
print(f"Worst absolute error: {float(np.max(np.abs(_pred - _true))):.4f}")
print("=" * 52)
if r2 >= TARGET_R2:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

Histogram gradient boosting handles the thresholds and interactions natively. The sweep fits three learning rates and lets early stopping on a validation split choose the number of iterations for each. The tradeoff shows in the printout: the smaller the learning rate, the more iterations it needs before validation stops improving, and the slightly better its validation score. All three land comfortably above the target.

```python
from sklearn.ensemble import HistGradientBoostingRegressor

model, best_score = None, -np.inf
for lr in [0.03, 0.08, 0.2]:
    candidate = HistGradientBoostingRegressor(
        learning_rate=lr, max_iter=500, early_stopping=True,
        validation_fraction=0.15, scoring="r2", random_state=0,
    )
    candidate.fit(X_train, y_train)
    score = candidate.validation_score_[-1]
    print(f"learning_rate={lr}: {candidate.n_iter_} iterations, validation R^2 {score:.4f}")
    if score > best_score:
        model, best_score = candidate, score

print(f"chosen learning_rate: {model.learning_rate}")
predictions = model.predict(X_test)
```
